In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import find_peaks

radius_values = np.concatenate([
    np.full(26, 0.0113226119),
    np.full(26, 0.0001132261),
    np.full(26, 0.0000011323)
])# m, radii needed to achieve Oh values

velocity_values = np.array([
    1.029478574, 1.455902561, 1.783109195, 2.058957148, 2.301984072, 
    2.414341264, 2.521697207, 2.624665668, 2.723744286, 2.819343187, 
    2.911805123, 3.00142002, 3.088435721, 3.173066068, 3.255497096, 
    4.603968145, 5.638686374, 6.510994191, 7.279512806, 10.29478574, 
    14.55902561, 17.83109195, 20.58957148, 23.01984072, 32.55497096, 
    46.03968145, 
    10.29478574, 14.55902561, 17.83109195, 20.58957148, 23.01984072, 
    24.14341264, 25.21697207, 26.24665668, 27.23744286, 28.19343187, 
    29.11805123, 30.0142002, 30.88435721, 31.73066068, 32.55497096, 
    46.03968145, 56.38686374, 65.10994191, 72.79512806, 102.9478574, 
    145.5902561, 178.3109195, 205.8957148, 230.1984072, 325.5497096, 
    460.3968145, 
    102.9478574, 145.5902561, 178.3109195, 205.8957148, 230.1984072, 
    241.4341264, 252.1697207, 262.4665668, 272.3744286, 281.9343187, 
    291.1805123, 300.142002, 308.8435721, 317.3066068, 325.5497096, 
    460.3968145, 563.8686374, 651.0994191, 727.9512806, 1029.478574, 
    1455.902561, 1783.109195, 2058.957148, 2301.984072, 3255.497096, 
    4603.968145
]) # m/s, velocity values

DUMMY CODE, FOR TESTING ODE SOLVERS

In [10]:
#Pick the We-Oh pair
k = 2

# All constants and initial conditions are defined here
y_0 = 0      # initial displacement (starts as a perfect sphere)
y_dot_0 = 0  # initial displacement velocity

Cf = 1/3     # force constant
Ck = 8       # linear constant
Cd = 5       # damping constant
Cb = 1/2     # breakup constant

# These values may be changed:
rho_g = 1.18  # kg/m^3, density of gaseous ethylene
rho_l = 997   # kg/m^3, density of liquid
r = radius_values[k]  # m, radius of droplet
sigma = 0.0708  # kg/s^2, surface tension
u = velocity_values[k]  # m/s, velocity

mu_l = 0.000894  # Pa*s, the dynamic viscosity of Jet A
t_bu = 8*np.sqrt(rho_l/rho_g)*(r/u)  # breakup time of particle

Oh = mu_l/np.sqrt(rho_l*r*sigma)  # Ohnesorge number
We = rho_g*(u**2)*r/sigma         # Weber number

omega = np.sqrt(Ck*sigma/(rho_l*r**3))  # oscillation frequency

duration = 10*(2*np.pi/omega)  # seconds, the time we want to watch this particle
len_t = 1000  # length of time vector

dt = np.linspace(0, duration, len_t)  # s, time vector
timestep = dt[1]-dt[0]  # timestep, in seconds

t_d = (2/Cd)*(rho_l*r**2/mu_l)

#------------------------------------------------
#---ORIGINAL METHOD OF SOLVING, FOR COMPARISON---
#------------------------------------------------
# define size of array beforehand
y_vals_original = np.zeros(len_t)  # array for normalized displacements over time
y_dot_vals_original = np.zeros(len_t)  # array for normalized change in displacement over time

y_vals_original[0] = y_0
y_dot_vals_original[0] = y_dot_0
volume = (4/3)*np.pi*r**3  # constant

breakup_time = 0  # Breakup time
breakup_iteration = len_t - 1

for i in range(len_t-1):
    # Defining some terms which show up frequently
    s = np.sin(omega*timestep)
    c = np.cos(omega*timestep)
    ex = np.exp(-timestep/t_d)
    y_We12 = y_vals_original[i] - We/12

    y_vals_original[i+1] = (We/12) + ex*((y_We12)*c + (1/omega)*(y_dot_vals_original[i] + (y_We12/t_d))*s)
    y_dot_vals_original[i+1] = (((We/12) - y_vals_original[i+1])/t_d) + omega*ex*((1/omega)*(y_dot_vals_original[i] + (y_We12/t_d))*c - (y_We12)*s)

#------------------------------------------------
#---USING ANALYTICAL METHOD---
#------------------------------------------------
from analytical_ODE_solver import analytical_ODE_solver
A = 5*rho_l/(rho_l*r**2) 
B = 8*sigma/(rho_l*r**3)
C = 2*rho_g*(u**2)/(rho_l*r**2)
dt = timestep
y0 = 0
y_dot0 = 0
t0 = 0
t1 = duration

t_analytical, y_analytical, y_dot_analytical = analytical_ODE_solver(A, B, C, dt, y0, y_dot0, t0, t1)

c:\Users\ishud\OneDrive - Georgia Institute of Technology\0 - Research OEF\Particle Modeling\analytical_ODE_solver.py:10: RuntimeWarning: invalid value encountered in sqrt
  omega = np.sqrt(B-((A**2)/4))
